In [0]:
import importlib.util

def ensure_package(package_name):
    spec = importlib.util.find_spec(package_name)
    if spec is None:
        print(f"installing {package_name}")
        %pip install {package_name}
        dbutils.library.restartPython()
    else:
        print(f"{package_name} already installed.")

ensure_package("databricks-zerobus-ingest-sdk")

In [0]:
# ── Zerobus configurations ────────────────────────────────────────────────
ZB_CLIENT_ID = dbutils.secrets.get(scope="skybound_scope", key="sp-skybound-client-id")
ZB_CLIENT_SECRET = dbutils.secrets.get(scope="skybound_scope", key="sp-skybound-client-secret")

ZB_SERVER = "https://7405605185991044.zerobus.westeurope.azuredatabricks.net"
ZB_WORKSPACE = "https://adb-7405605185991044.4.azuredatabricks.net"

CATALOG = dbutils.widgets.get("catalog")
TABLE_NAME = f"{CATALOG}.skybound_bronze.weather_reports_bronze"

In [0]:
import logging
import requests
import time
import threading
from datetime import datetime, timezone
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties, AckCallback

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("metar_producer")

# ── Acknowledgement callback ──────────────────────────────────────────────
class MetarAckCallback(AckCallback):
    def __init__(self):
        self.acked = 0
        self.errors = 0

    def on_ack(self, offset: int) -> None:
        self.acked += 1
        if self.acked % 50 == 0:
            logger.info(f"Acknowledged {self.acked} records (offset: {offset})")

    def on_error(self, offset: int, error_message: str) -> None:
        self.errors += 1
        logger.error(f"Ingestion error at offset {offset}: {error_message}")


# ── Zerobus stream setup ──────────────────────────────────────────────────
ack_callback = MetarAckCallback()

def create_zerobus_stream():
    """Create a new ZeroBus stream with AckCallback."""
    sdk = ZerobusSdk(ZB_SERVER, ZB_WORKSPACE)
    table_properties = TableProperties(TABLE_NAME)
    options = StreamConfigurationOptions(
        record_type=RecordType.JSON,
        ack_callback=ack_callback
    )
    stream = sdk.create_stream(ZB_CLIENT_ID, ZB_CLIENT_SECRET, table_properties, options)
    time.sleep(1)  
    return stream

def close_stream_with_timeout(stream, timeout_sec=5):
    t = threading.Thread(target=stream.close)
    t.start()
    t.join(timeout=timeout_sec)
    if t.is_alive():
        logger.warning(f"stream.close() did not complete within {timeout_sec}s, moving on")


In [0]:
# ── API connection details ────────────────────────────────────────────────

API_URL = "https://aviationweather.gov/api/data/metar"
EUROPE_BBOX = "34,-24,71,40"

# ── API functions ──────────────────────────────────────────────────────────

def fetch_metars() -> list:
    try:
        resp = requests.get(
            API_URL,
            params={"bbox": EUROPE_BBOX, "format": "json", "hoursBeforeNow": 1},
            headers={"User-Agent": "aviation-analytics-demo/1.0 student-project"},
            timeout=30,
        )
        if resp.status_code == 204:
            logger.info(f"AWC returned 204 — no data available this cycle: {_ingestion_timestamp}")
            return []
        
        resp.raise_for_status()
        data = resp.json()
        logger.info(f"Fetched {len(data)} METAR records from AWC: {_ingestion_timestamp}")
        return data
    
    except Exception as e:
        logger.error(f"AWS API error: {e}")
        return []


def extract_sky_cover(clouds: list) -> tuple:
    if not clouds:
        return None, None
    severity = {"OVC": 6, "OVX": 5, "BKN": 4, "SCT": 3, "FEW": 2, "CLR": 1, "SKC": 1}
    dominant = max(clouds, key=lambda c: severity.get(c.get("cover", ""), 0))
    return dominant.get("cover"), dominant.get("base")


def stringify_record(obj):
    if isinstance(obj, dict):
        return {k: stringify_record(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [stringify_record(v) for v in obj]
    elif obj is None:
        return None
    else:
        return str(obj)


def build_record(metar: dict, _ingestion_timestamp: str) -> dict:
    sky_cover, cloud_base_ft = extract_sky_cover(metar.get("clouds", []))

    record = {k: v for k, v in metar.items() if k != "clouds"}
    record["sky_cover"] = sky_cover
    record["cloud_base_ft"] = cloud_base_ft
    record["_ingestion_timestamp"] = _ingestion_timestamp

    return stringify_record(record)


In [0]:
# ── Single run ────────────────────────────────────────────────────────────

_ingestion_timestamp = datetime.now(timezone.utc).isoformat()

metars = fetch_metars()

if not metars:
    logger.info("Nothing to ingest from AWC — exiting")
else:
    stream = create_zerobus_stream()
    sent = 0
    skipped = 0
    try:
        for metar in metars:
            try:
                record = build_record(metar, _ingestion_timestamp)
                stream.ingest_record_offset(record)
                sent += 1
            except Exception as rec_err:
                skipped += 1
                if skipped <= 3:
                    logger.warning(
                        f"Skipped bad record "
                        f"(icaoId={metar.get('icaoId', '?')}): {rec_err}"
                    )
    finally:
        close_stream_with_timeout(stream, timeout_sec=5)

    logger.info(
        f"Done: sent={sent}, skipped={skipped}, "
        f"acked={ack_callback.acked}, errors={ack_callback.errors} "
    )
